In [1]:
import csv
import math
import html
import matplotlib.pyplot as plt

from io import StringIO
from pathlib import Path

from IPython.display import display, FileLink
from ipywidgets import (
    Accordion,
    Button,
    Dropdown,
    FileUpload,
    HBox,
    HTML,
    Layout,
    Output,
    RadioButtons,
    Textarea,
    VBox
)

try:
    from pyproj import Geod
except ImportError as exc:
    raise ImportError(
        "The WGS84 method requires pyproj.\n"
        "Install it in a Jupyter Notebook with:\n\n"
        "%pip install pyproj"
    ) from exc


# ================================================================
# CONSTANTS
# ================================================================

EPS = 1e-12

MPS_TO_KMH = 3.6
MPS_TO_MPH = 2.2369362920544

M_TO_KM = 1.0 / 1000.0
M_TO_MI = 1.0 / 1609.344
M_TO_FT = 3.2808398950131

WGS84_GEOD = Geod(ellps="WGS84")

OUTPUT_FILE = "results.csv"


# ================================================================
# GENERAL VALIDATION
# ================================================================

def validate_lat_lon(latitude, longitude, row_number):
    """
    Validate one latitude/longitude coordinate pair.
    """

    if not math.isfinite(latitude):
        raise ValueError(
            f"CSV row {row_number}: latitude must be a finite number."
        )

    if not math.isfinite(longitude):
        raise ValueError(
            f"CSV row {row_number}: longitude must be a finite number."
        )

    if not -90.0 <= latitude <= 90.0:
        raise ValueError(
            f"CSV row {row_number}: latitude {latitude} is outside "
            "the valid range of -90 to 90 degrees."
        )

    if not -180.0 <= longitude <= 180.0:
        raise ValueError(
            f"CSV row {row_number}: longitude {longitude} is outside "
            "the valid range of -180 to 180 degrees."
        )


def parse_float(value, column_name, row_number):
    """
    Convert a CSV value to a finite floating-point number.
    """

    value_text = str(value).strip()

    if value_text == "":
        raise ValueError(
            f"CSV row {row_number}: column '{column_name}' is empty."
        )

    try:
        number = float(value_text)
    except ValueError as exc:
        raise ValueError(
            f"CSV row {row_number}: '{value_text}' in column "
            f"'{column_name}' is not numeric."
        ) from exc

    if not math.isfinite(number):
        raise ValueError(
            f"CSV row {row_number}: column '{column_name}' "
            "must contain a finite number."
        )

    return number


def normalize_longitude_difference(delta_longitude_deg):
    """
    Normalize a longitude difference to [-180, 180).

    This prevents a date-line crossing from being interpreted as
    nearly 360 degrees of movement.
    """

    return (delta_longitude_deg + 180.0) % 360.0 - 180.0


# ================================================================
# LOCAL CARTESIAN METHOD
# ================================================================

def compute_local_scale_factors(latitude_deg):
    """
    Estimate meters per degree of latitude and longitude.

    The equations vary the scale factors by latitude and approximate
    the curvature of the WGS84 ellipsoid.

    Returns
    -------
    meters_per_degree_latitude
    meters_per_degree_longitude
    """

    latitude_rad = math.radians(latitude_deg)

    meters_per_degree_latitude = (
        111132.92
        - 559.82 * math.cos(2.0 * latitude_rad)
        + 1.175 * math.cos(4.0 * latitude_rad)
        - 0.0023 * math.cos(6.0 * latitude_rad)
    )

    meters_per_degree_longitude = (
        111412.84 * math.cos(latitude_rad)
        - 93.5 * math.cos(3.0 * latitude_rad)
        + 0.118 * math.cos(5.0 * latitude_rad)
    )

    return (
        meters_per_degree_latitude,
        meters_per_degree_longitude
    )


def local_cartesian_distance_m(lat1, lon1, lat2, lon2):
    """
    Calculate approximate distance using a local Cartesian plane.

    GPS degree differences are converted to local east-west and
    north-south distances in meters. The Pythagorean theorem is then
    used to calculate the segment distance.

    This is a local planar approximation. It is not a complete formal
    Orthographic map projection.
    """

    delta_latitude_deg = lat2 - lat1

    delta_longitude_deg = normalize_longitude_difference(
        lon2 - lon1
    )

    # Use the midpoint latitude for the segment's scale factors.
    midpoint_latitude_deg = (lat1 + lat2) / 2.0

    (
        meters_per_degree_latitude,
        meters_per_degree_longitude
    ) = compute_local_scale_factors(midpoint_latitude_deg)

    # East-west displacement.
    dx_m = (
        delta_longitude_deg
        * meters_per_degree_longitude
    )

    # North-south displacement.
    dy_m = (
        delta_latitude_deg
        * meters_per_degree_latitude
    )

    # sqrt(dx^2 + dy^2)
    return math.hypot(dx_m, dy_m)


# ================================================================
# WGS84 GEODESIC METHOD
# ================================================================

def wgs84_geodesic_distance_m(lat1, lon1, lat2, lon2):
    """
    Calculate ellipsoidal geodesic distance on WGS84.

    pyproj.Geod.inv returns:
        forward azimuth,
        reverse azimuth,
        distance in meters

    Geod.inv expects longitude before latitude.
    """

    _, _, distance_m = WGS84_GEOD.inv(
        lon1,
        lat1,
        lon2,
        lat2
    )

    return abs(distance_m)


def calculate_segment_distance_m(
    lat1,
    lon1,
    lat2,
    lon2,
    method
):
    """
    Calculate one segment's distance using the selected method.
    """

    delta_latitude = lat2 - lat1

    delta_longitude = normalize_longitude_difference(
        lon2 - lon1
    )

    if (
        abs(delta_latitude) < EPS
        and abs(delta_longitude) < EPS
    ):
        return 0.0

    if method == "wgs84":
        return wgs84_geodesic_distance_m(
            lat1,
            lon1,
            lat2,
            lon2
        )

    if method == "local":
        return local_cartesian_distance_m(
            lat1,
            lon1,
            lat2,
            lon2
        )

    raise ValueError(
        f"Unknown distance method: {method}"
    )


def get_method_label(method):
    """
    Return the display name for the selected method.
    """

    if method == "wgs84":
        return "WGS84 ellipsoidal geodesic"

    if method == "local":
        return "Local Cartesian approximation"

    return str(method)


# ================================================================
# CSV PROCESSING
# ================================================================

def process_csv(
    input_data,
    lat_col,
    lon_col,
    time_col,
    method
):
    """
    Process GPS CSV data.

    The function calculates:
      - relative time
      - segment duration
      - segment distance
      - cumulative distance
      - speed
    """

    csv_file = StringIO(input_data)
    reader = csv.DictReader(csv_file)

    if not reader.fieldnames:
        raise ValueError(
            "The CSV does not contain a header row."
        )

    original_headers = [
        str(header).strip()
        for header in reader.fieldnames
        if header is not None
    ]

    if any(header == "" for header in original_headers):
        raise ValueError(
            "One or more CSV column headers are empty."
        )

    if len(set(original_headers)) != len(original_headers):
        raise ValueError(
            "The CSV contains duplicate column names after "
            "leading and trailing spaces are removed."
        )

    rows = []

    for original_row in reader:
        cleaned_row = {
            str(key).strip(): (
                value.strip()
                if isinstance(value, str)
                else value
            )
            for key, value in original_row.items()
            if key is not None
        }

        # Ignore completely blank rows.
        if not any(
            str(value).strip()
            for value in cleaned_row.values()
            if value is not None
        ):
            continue

        rows.append(cleaned_row)

    if len(rows) < 2:
        raise ValueError(
            "At least two GPS data rows are required to "
            "calculate distance and speed."
        )

    selected_columns = [
        lat_col,
        lon_col,
        time_col
    ]

    if len(set(selected_columns)) != 3:
        raise ValueError(
            "Latitude, longitude, and timestamp must be "
            "three different CSV columns."
        )

    for selected_column in selected_columns:
        if selected_column not in original_headers:
            raise ValueError(
                f"Selected column '{selected_column}' "
                "was not found in the CSV."
            )

    normalized_rows = []

    # CSV row 1 contains headers, so data begins on row 2.
    for csv_row_number, row in enumerate(rows, start=2):
        latitude = parse_float(
            row.get(lat_col, ""),
            lat_col,
            csv_row_number
        )

        longitude = parse_float(
            row.get(lon_col, ""),
            lon_col,
            csv_row_number
        )

        timestamp = parse_float(
            row.get(time_col, ""),
            time_col,
            csv_row_number
        )

        validate_lat_lon(
            latitude,
            longitude,
            csv_row_number
        )

        normalized_rows.append(
            {
                "row": row,
                "latitude": latitude,
                "longitude": longitude,
                "timestamp": timestamp,
                "csv_row_number": csv_row_number
            }
        )

    added_headers = [
        "calculation_method",
        "relative_time_s",
        "segment_duration_s",
        "segment_distance_m",
        "segment_distance_ft",
        "segment_distance_km",
        "segment_distance_mi",
        "cumulative_distance_m",
        "cumulative_distance_ft",
        "cumulative_distance_km",
        "cumulative_distance_mi",
        "speed_mps",
        "speed_ftps",
        "speed_kmh",
        "speed_mph"
    ]

    # Remove old calculated columns if a previous results file is
    # pasted or uploaded and processed again.
    output_headers = [
        header
        for header in original_headers
        if header not in added_headers
    ] + added_headers

    method_label = get_method_label(method)

    first_timestamp = normalized_rows[0]["timestamp"]

    cumulative_distance_m = 0.0

    plot_times = []
    plot_speeds_mph = []

    for index, item in enumerate(normalized_rows):
        row = item["row"]
        timestamp = item["timestamp"]

        relative_time_s = timestamp - first_timestamp

        row["calculation_method"] = method_label
        row["relative_time_s"] = f"{relative_time_s:.3f}"

        # First data point is the starting position.
        if index == 0:
            row["segment_duration_s"] = "0.000"

            row["segment_distance_m"] = "0.00"
            row["segment_distance_ft"] = "0.00"
            row["segment_distance_km"] = "0.000000"
            row["segment_distance_mi"] = "0.000000"

            row["cumulative_distance_m"] = "0.00"
            row["cumulative_distance_ft"] = "0.00"
            row["cumulative_distance_km"] = "0.000000"
            row["cumulative_distance_mi"] = "0.000000"

            row["speed_mps"] = "0.00"
            row["speed_ftps"] = "0.00"
            row["speed_kmh"] = "0.00"
            row["speed_mph"] = "0.00"

            plot_times.append(relative_time_s)
            plot_speeds_mph.append(0.0)

            continue

        previous = normalized_rows[index - 1]

        delta_time_s = (
            timestamp
            - previous["timestamp"]
        )

        if delta_time_s <= 0.0:
            raise ValueError(
                f"CSV row {item['csv_row_number']}: timestamp "
                "must be greater than the preceding timestamp. "
                f"Previous timestamp={previous['timestamp']}; "
                f"current timestamp={timestamp}."
            )

        segment_distance_m = calculate_segment_distance_m(
            previous["latitude"],
            previous["longitude"],
            item["latitude"],
            item["longitude"],
            method
        )

        cumulative_distance_m += segment_distance_m

        speed_mps = (
            segment_distance_m
            / delta_time_s
        )

        speed_ftps = speed_mps * M_TO_FT
        speed_kmh = speed_mps * MPS_TO_KMH
        speed_mph = speed_mps * MPS_TO_MPH

        row["segment_duration_s"] = (
            f"{delta_time_s:.3f}"
        )

        row["segment_distance_m"] = (
            f"{segment_distance_m:.2f}"
        )

        row["segment_distance_ft"] = (
            f"{segment_distance_m * M_TO_FT:.2f}"
        )

        row["segment_distance_km"] = (
            f"{segment_distance_m * M_TO_KM:.6f}"
        )

        row["segment_distance_mi"] = (
            f"{segment_distance_m * M_TO_MI:.6f}"
        )

        row["cumulative_distance_m"] = (
            f"{cumulative_distance_m:.2f}"
        )

        row["cumulative_distance_ft"] = (
            f"{cumulative_distance_m * M_TO_FT:.2f}"
        )

        row["cumulative_distance_km"] = (
            f"{cumulative_distance_m * M_TO_KM:.6f}"
        )

        row["cumulative_distance_mi"] = (
            f"{cumulative_distance_m * M_TO_MI:.6f}"
        )

        row["speed_mps"] = (
            f"{speed_mps:.2f}"
        )

        row["speed_ftps"] = (
            f"{speed_ftps:.2f}"
        )

        row["speed_kmh"] = (
            f"{speed_kmh:.2f}"
        )

        row["speed_mph"] = (
            f"{speed_mph:.2f}"
        )

        plot_times.append(relative_time_s)
        plot_speeds_mph.append(speed_mph)

    output_path = Path(OUTPUT_FILE)

    with output_path.open(
        "w",
        newline="",
        encoding="utf-8"
    ) as output_file:
        writer = csv.DictWriter(
            output_file,
            fieldnames=output_headers,
            extrasaction="ignore"
        )

        writer.writeheader()

        writer.writerows(
            item["row"]
            for item in normalized_rows
        )

    total_distance_ft = (
        cumulative_distance_m
        * M_TO_FT
    )

    total_distance_km = (
        cumulative_distance_m
        * M_TO_KM
    )

    total_distance_mi = (
        cumulative_distance_m
        * M_TO_MI
    )

    valid_speeds = [
        speed
        for speed in plot_speeds_mph[1:]
        if math.isfinite(speed)
    ]

    maximum_speed_mph = (
        max(valid_speeds)
        if valid_speeds
        else 0.0
    )

    total_elapsed_time_s = (
        normalized_rows[-1]["timestamp"]
        - normalized_rows[0]["timestamp"]
    )

    average_speed_mph = (
        total_distance_mi
        / (total_elapsed_time_s / 3600.0)
        if total_elapsed_time_s > 0.0
        else 0.0
    )

    print("Processing complete.")
    print(f"Method: {method_label}")
    print(f"Rows processed: {len(normalized_rows):,}")
    print(
        f"Elapsed time: {total_elapsed_time_s:.3f} seconds"
    )
    print(
        f"Total distance: {cumulative_distance_m:.2f} m"
    )
    print(
        f"Total distance: {total_distance_ft:.2f} ft"
    )
    print(
        f"Total distance: {total_distance_km:.6f} km"
    )
    print(
        f"Total distance: {total_distance_mi:.6f} mi"
    )
    print(
        f"Average track speed: {average_speed_mph:.2f} mph"
    )
    print(
        f"Maximum interval speed: {maximum_speed_mph:.2f} mph"
    )
    print(f"Results saved to: {output_path.resolve()}")

    display(
        FileLink(
            str(output_path),
            result_html_prefix=(
                "<b>Download calculated results: </b>"
            )
        )
    )

    plt.figure(figsize=(11, 5))

    plt.plot(
        plot_times,
        plot_speeds_mph,
        marker="o",
        linewidth=1.5,
        markersize=4
    )

    plt.title(
        f"GPS Interval Speed\n{method_label}"
    )

    plt.xlabel("Relative time (seconds)")
    plt.ylabel("Speed (mph)")
    plt.grid(True, alpha=0.35)
    plt.tight_layout()
    plt.show()


# ================================================================
# FILE UPLOAD FUNCTIONS
# ================================================================

def extract_uploaded_file(upload_value):
    """
    Extract one file name and its content from FileUpload.

    Supports common ipywidgets 7 and ipywidgets 8 FileUpload
    representations.
    """

    if not upload_value:
        return None, None

    # ipywidgets 8 normally returns a tuple of dictionaries.
    if isinstance(upload_value, (tuple, list)):
        uploaded_file = upload_value[0]

        file_name = uploaded_file.get(
            "name",
            "uploaded.csv"
        )

        file_content = uploaded_file.get("content")

        return file_name, file_content

    # ipywidgets 7 normally returns a dictionary.
    if isinstance(upload_value, dict):
        # Direct uploaded-file dictionary.
        if "content" in upload_value:
            file_name = upload_value.get(
                "name",
                "uploaded.csv"
            )

            return (
                file_name,
                upload_value["content"]
            )

        # Dictionary keyed by file name.
        file_name = next(iter(upload_value))
        uploaded_file = upload_value[file_name]

        return (
            file_name,
            uploaded_file.get("content")
        )

    raise ValueError(
        "The uploaded file format was not recognized."
    )


def decode_csv_content(file_content):
    """
    Decode uploaded file content into text.

    Decoding order:
      1. UTF-8 with optional byte-order mark
      2. Windows-1252
      3. Latin-1
    """

    if file_content is None:
        raise ValueError(
            "The uploaded file does not contain readable data."
        )

    if isinstance(file_content, memoryview):
        file_content = file_content.tobytes()

    if isinstance(file_content, bytearray):
        file_content = bytes(file_content)

    if isinstance(file_content, str):
        return file_content

    if not isinstance(file_content, bytes):
        raise ValueError(
            "The uploaded file content is not in a supported format."
        )

    encodings = [
        "utf-8-sig",
        "windows-1252",
        "latin-1"
    ]

    for encoding in encodings:
        try:
            return file_content.decode(encoding)
        except UnicodeDecodeError:
            continue

    raise ValueError(
        "The CSV file could not be decoded. Save the file "
        "as a UTF-8 CSV and upload it again."
    )


def inspect_csv_headers(csv_text):
    """
    Read and clean the first CSV row.
    """

    preview_reader = csv.reader(
        StringIO(csv_text)
    )

    raw_headers = next(
        preview_reader,
        None
    )

    if not raw_headers:
        raise ValueError(
            "The CSV does not contain a header row."
        )

    headers = [
        str(header).strip()
        for header in raw_headers
    ]

    if any(header == "" for header in headers):
        raise ValueError(
            "One or more CSV column headers are empty."
        )

    if len(set(headers)) != len(headers):
        raise ValueError(
            "The CSV contains duplicate column names."
        )

    return headers


def validate_uploaded_csv(file_name, csv_text):
    """
    Perform basic validation before copying an uploaded CSV file
    into the text area.
    """

    if not file_name.lower().endswith(".csv"):
        raise ValueError(
            "Please upload a file with a .csv extension."
        )

    if not csv_text.strip():
        raise ValueError(
            "The uploaded CSV file is empty."
        )

    headers = inspect_csv_headers(csv_text)

    if len(headers) < 3:
        raise ValueError(
            "The CSV must have at least three columns for "
            "latitude, longitude, and timestamp."
        )

    return headers


# ================================================================
# COLUMN DETECTION
# ================================================================

def find_default_column(
    headers,
    possible_names,
    fallback_index
):
    """
    Try to identify a CSV column from common names.

    If no name matches, return a column based on the supplied
    fallback position.
    """

    lowercase_lookup = {
        header.lower(): header
        for header in headers
    }

    for possible_name in possible_names:
        if possible_name in lowercase_lookup:
            return lowercase_lookup[possible_name]

    safe_index = min(
        fallback_index,
        len(headers) - 1
    )

    return headers[safe_index]


def determine_column_defaults(headers):
    """
    Return likely latitude, longitude, and timestamp columns.
    """

    latitude_default = find_default_column(
        headers,
        [
            "lat",
            "latitude",
            "gps_lat",
            "gps latitude",
            "gps_latitude",
            "latitude_deg",
            "latitude degrees"
        ],
        0
    )

    longitude_default = find_default_column(
        headers,
        [
            "lon",
            "long",
            "lng",
            "longitude",
            "gps_lon",
            "gps longitude",
            "gps_longitude",
            "longitude_deg",
            "longitude degrees"
        ],
        1
    )

    time_default = find_default_column(
        headers,
        [
            "time",
            "timestamp",
            "seconds",
            "time_s",
            "relative_time",
            "relative time",
            "elapsed_time",
            "elapsed time",
            "elapsed_seconds"
        ],
        2
    )

    return (
        latitude_default,
        longitude_default,
        time_default
    )


# ================================================================
# HELP CONTENT
# ================================================================

HELP_HTML = """
<div style="
    padding: 14px;
    line-height: 1.55;
    max-width: 1100px;
">
    <h2>GPS Distance and Speed Calculator Help</h2>

    <h3>1. Purpose</h3>

    <p>
        This calculator reads consecutive GPS observations containing
        latitude, longitude, and numeric timestamps. For each pair of
        consecutive points, the calculator determines distance,
        elapsed time, and interval speed.
    </p>

    <ol>
        <li>Read two consecutive GPS coordinates.</li>
        <li>Calculate the distance between the coordinates.</li>
        <li>Subtract the timestamps to obtain elapsed time.</li>
        <li>Divide segment distance by elapsed time.</li>
        <li>Add the segment distance to cumulative distance.</li>
    </ol>

    <h3>2. CSV input options</h3>

    <p>The calculator supports three input workflows:</p>

    <ul>
        <li>
            <b>Upload CSV:</b> Select a CSV file from the computer.
            The entire file is copied into the editable text area.
        </li>
        <li>
            <b>Paste manually:</b> Paste CSV text directly into the
            text area.
        </li>
        <li>
            <b>Load example:</b> Load the built-in example CSV.
        </li>
    </ul>

    <p>
        Because uploaded data is copied into the text area, the data
        can be reviewed or edited before processing.
    </p>

    <h3>3. Required CSV columns</h3>

    <p>The CSV must include at least:</p>

    <ul>
        <li>A latitude column in decimal degrees.</li>
        <li>A longitude column in decimal degrees.</li>
        <li>A numeric timestamp column.</li>
    </ul>

    <p>Example:</p>

    <pre style="white-space: pre-wrap;">
time,lat,lon
0,40.0000,-105.0000
1,40.0005,-105.0003
2,40.0011,-105.0007
3,40.0016,-105.0010
    </pre>

    <p>
        Timestamp differences are assumed to be seconds. Timestamps
        must increase from one data row to the next.
    </p>

    <h3>4. Latitude and longitude</h3>

    <p>
        Latitude indicates north-south angular position. Valid decimal
        latitude values range from -90 to +90 degrees.
    </p>

    <p>
        Longitude indicates east-west angular position. Valid decimal
        longitude values range from -180 to +180 degrees.
    </p>

    <p>
        Latitude and longitude are angular coordinates, not direct
        linear distances. One degree of longitude represents less
        ground distance as a location approaches either pole.
    </p>

    <h3>5. WGS84 ellipsoidal geodesic</h3>

    <p>
        WGS84 means <b>World Geodetic System 1984</b>. WGS84 is the
        reference coordinate system commonly used by GPS.
    </p>

    <p>
        WGS84 approximates Earth as a flattened ellipsoid. Earth is
        slightly wider around the equator than the distance between
        the geographic poles.
    </p>

    <p>
        A <b>geodesic</b> is the shortest surface path between two
        coordinates on the selected Earth model.
    </p>

    <p>The program calls:</p>

    <pre style="white-space: pre-wrap;">
forward_azimuth, reverse_azimuth, distance_m =
    WGS84_GEOD.inv(lon1, lat1, lon2, lat2)
    </pre>

    <p>
        The function returns forward direction, reverse direction, and
        ellipsoidal surface distance in meters. Only distance is needed
        to calculate speed.
    </p>

    <p>
        <b>WGS84 geodesic is not a map projection.</b> A map projection
        transforms curved Earth coordinates into flat map coordinates.
        This option calculates distance directly on the WGS84 ellipsoid.
    </p>

    <p><b>Benefits:</b></p>

    <ul>
        <li>Appropriate for short and long segments.</li>
        <li>Accounts for Earth's ellipsoidal shape.</li>
        <li>Works globally.</li>
        <li>Does not require a manually selected reference latitude.</li>
        <li>Recommended for final or defensible distance results.</li>
    </ul>

    <h3>6. Local Cartesian approximation</h3>

    <p>
        The local method treats each short segment as movement on a
        flat x-y plane. The program calculates latitude and longitude
        differences:
    </p>

    <pre style="white-space: pre-wrap;">
delta_latitude = latitude_2 - latitude_1
delta_longitude = longitude_2 - longitude_1
    </pre>

    <p>
        The program calculates latitude-dependent scale factors at the
        midpoint latitude of each segment.
    </p>

    <pre style="white-space: pre-wrap;">
dx = delta_longitude * meters_per_degree_longitude
dy = delta_latitude * meters_per_degree_latitude
    </pre>

    <p>
        The east-west displacement is dx. The north-south displacement
        is dy.
    </p>

    <p>The distance is calculated with:</p>

    <pre style="white-space: pre-wrap;">
distance = sqrt(dx^2 + dy^2)
    </pre>

    <p>
        This is the Pythagorean theorem applied to a short local
        segment.
    </p>

    <p>
        This method is a local Cartesian approximation. It is
        <b>not a complete formal Orthographic map projection</b>.
        A formal Orthographic projection requires a projection center
        and specific spherical or ellipsoidal projection equations.
    </p>

    <p><b>Benefits:</b></p>

    <ul>
        <li>Simple calculation.</li>
        <li>Computationally inexpensive.</li>
        <li>Easy to interpret as local x-y movement.</li>
        <li>Highly effective for closely spaced vehicle observations.</li>
    </ul>

    <p><b>Limitations:</b></p>

    <ul>
        <li>It is an approximation.</li>
        <li>Error increases as individual segments become longer.</li>
        <li>It should not be treated as one global flat coordinate system.</li>
        <li>Special care is required near the geographic poles.</li>
    </ul>

    <h3>7. Which method should be selected?</h3>

    <p>
        Select <b>WGS84 ellipsoidal geodesic</b> when global validity,
        accuracy, consistency, or defensibility is most important.
        WGS84 is the default and generally recommended option.
    </p>

    <p>
        Select <b>Local Cartesian approximation</b> for closely spaced
        vehicle GPS observations when a transparent local planar
        calculation is desired.
    </p>

    <p>
        For one-second vehicle points separated by several meters, the
        difference between the two methods will normally be much smaller
        than ordinary consumer or vehicle GPS position noise.
    </p>

    <h3>8. Distance formulas</h3>

    <p><b>Local Cartesian distance:</b></p>

    <pre style="white-space: pre-wrap;">
d = sqrt(dx^2 + dy^2)
    </pre>

    <p><b>Cumulative distance:</b></p>

    <pre style="white-space: pre-wrap;">
cumulative_distance[i] =
    cumulative_distance[i - 1] + segment_distance[i]
    </pre>

    <p>
        Cumulative distance is the sum of all point-to-point segments.
        It may differ from road-network distance or vehicle odometer
        distance.
    </p>

    <h3>9. Speed formulas</h3>

    <pre style="white-space: pre-wrap;">
delta_time_s = timestamp_2 - timestamp_1
speed_mps = segment_distance_m / delta_time_s
speed_ftps = speed_mps * 3.280839895
speed_kmh = speed_mps * 3.6
speed_mph = speed_mps * 2.236936292
    </pre>

    <p>
        The calculated speed is an interval speed between two GPS
        coordinates. It is not the same as instantaneous GPS Doppler
        speed reported directly by some GPS receivers.
    </p>

    <h3>10. Output columns</h3>

    <ul>
        <li>
            <b>calculation_method:</b> Selected distance method.
        </li>
        <li>
            <b>relative_time_s:</b> Seconds since the first timestamp.
        </li>
        <li>
            <b>segment_duration_s:</b> Seconds since the preceding row.
        </li>
        <li>
            <b>segment_distance:</b> Distance from the preceding point.
        </li>
        <li>
            <b>cumulative_distance:</b> Sum of completed segment distances.
        </li>
        <li>
            <b>speed:</b> Segment distance divided by segment duration.
        </li>
    </ul>

    <h3>11. GPS measurement error</h3>

    <p>
        Selecting a more accurate distance formula does not eliminate
        location measurement error. GPS coordinates may move while a
        vehicle is stationary because of receiver error, satellite
        geometry, obstructed sky view, coordinate rounding, and signal
        reflections from buildings or other surfaces.
    </p>

    <p>
        For example, a false 3-meter position change over one second
        produces:
    </p>

    <pre style="white-space: pre-wrap;">
3 meters / 1 second = 3 m/s
3 m/s = approximately 6.71 mph
    </pre>

    <p>
        For noisy data, consider adding a stationary threshold, maximum
        plausible speed, GPS accuracy filter, or smoothing procedure.
    </p>

    <h3>12. Date-line handling</h3>

    <p>
        The local method normalizes longitude differences to the range
        -180 to +180 degrees. This prevents a date-line crossing from
        being interpreted as nearly 360 degrees of east-west movement.
    </p>

    <h3>13. Results file</h3>

    <p>
        Processed output is written to <b>results.csv</b>. A download
        link appears after successful processing.
    </p>
</div>
"""


# ================================================================
# USER INTERFACE
# ================================================================

def run_with_ui():
    """
    Display and operate the complete notebook interface.
    """

    output = Output()

    input_box = Textarea(
        placeholder=(
            "Upload a CSV file, paste CSV data here, "
            "or select Load Example CSV..."
        ),
        layout=Layout(
            width="100%",
            height="240px"
        )
    )

    upload_button = FileUpload(
        accept=".csv,text/csv",
        multiple=False,
        description="Upload CSV",
        icon="upload",
        button_style="primary",
        layout=Layout(width="160px")
    )

    example_button = Button(
        description="Load Example",
        button_style="info",
        icon="file",
        layout=Layout(width="160px")
    )

    clear_button = Button(
        description="Clear",
        button_style="warning",
        icon="trash",
        layout=Layout(width="120px")
    )

    process_button = Button(
        description="Process CSV",
        button_style="success",
        icon="play",
        layout=Layout(width="170px")
    )

    method_selector = RadioButtons(
        options=[
            (
                "WGS84 ellipsoidal geodesic — recommended",
                "wgs84"
            ),
            (
                "Local Cartesian approximation — short segments",
                "local"
            )
        ],
        value="wgs84",
        description="Method:",
        disabled=False,
        layout=Layout(width="100%")
    )

    upload_status = HTML(
        value=(
            "<span style='color:#555;'>"
            "No CSV file uploaded."
            "</span>"
        )
    )

    input_status = HTML(
        value=""
    )

    example_csv = """time,lat,lon
0,40.0000,-105.0000
1,40.0005,-105.0003
2,40.0011,-105.0007
3,40.0016,-105.0010
"""

    help_section = Accordion(
        children=[
            HTML(HELP_HTML)
        ],
        selected_index=None,
        layout=Layout(width="100%")
    )

    help_section.set_title(
        0,
        "Help: methods, formulas, terminology, input, and output"
    )

    # ------------------------------------------------------------
    # Upload handling
    # ------------------------------------------------------------

    def handle_upload(change):
        """
        Decode the uploaded CSV and copy it into the text area.
        """

        with output:
            output.clear_output()

            try:
                file_name, file_content = extract_uploaded_file(
                    upload_button.value
                )

                if file_name is None:
                    upload_status.value = (
                        "<span style='color:#555;'>"
                        "No CSV file uploaded."
                        "</span>"
                    )
                    return

                csv_text = decode_csv_content(
                    file_content
                )

                headers = validate_uploaded_csv(
                    file_name,
                    csv_text
                )

                # Copy the complete file into the editable text area.
                input_box.value = csv_text

                data_rows = list(
                    csv.reader(
                        StringIO(csv_text)
                    )
                )

                total_lines = len(
                    csv_text.splitlines()
                )

                estimated_data_rows = max(
                    len(data_rows) - 1,
                    0
                )

                safe_file_name = html.escape(
                    file_name
                )

                safe_headers = ", ".join(
                    html.escape(header)
                    for header in headers
                )

                upload_status.value = (
                    "<div style='color:#137333;'>"
                    f"<b>{safe_file_name}</b> loaded successfully.<br>"
                    f"Detected {len(headers):,} columns and "
                    f"approximately {estimated_data_rows:,} data rows.<br>"
                    "The file content was copied into the editable "
                    "text area."
                    "</div>"
                )

                input_status.value = (
                    "<div style='color:#555;'>"
                    f"<b>Detected columns:</b> {safe_headers}"
                    "</div>"
                )

                print(f"Uploaded file: {file_name}")
                print(
                    f"Text lines loaded: {total_lines:,}"
                )
                print(
                    "The CSV content was copied into the text area."
                )
                print(
                    "Review or edit the data, select a method, "
                    "and click Process CSV."
                )

            except Exception as error:
                upload_status.value = (
                    "<div style='color:#b00020;'>"
                    f"<b>Upload failed:</b> "
                    f"{html.escape(str(error))}"
                    "</div>"
                )

                input_status.value = ""

                print(f"Upload error: {error}")

    upload_button.observe(
        handle_upload,
        names="value"
    )

    # ------------------------------------------------------------
    # Example data
    # ------------------------------------------------------------

    def load_example(_):
        """
        Copy the example CSV into the text area.
        """

        input_box.value = example_csv

        upload_status.value = (
            "<span style='color:#137333;'>"
            "Example CSV loaded into the text area."
            "</span>"
        )

        input_status.value = (
            "<span style='color:#555;'>"
            "<b>Detected columns:</b> time, lat, lon"
            "</span>"
        )

        with output:
            output.clear_output()

            print("Example CSV loaded.")
            print(
                "Select a distance method and click Process CSV."
            )

    example_button.on_click(
        load_example
    )

    # ------------------------------------------------------------
    # Clear data
    # ------------------------------------------------------------

    def clear_all(_):
        """
        Clear the text area, upload state, statuses, and output.
        """

        input_box.value = ""

        # Reset FileUpload. Widget behavior differs by version.
        try:
            upload_button.value = ()
        except Exception:
            try:
                upload_button.value.clear()
            except Exception:
                pass

            try:
                upload_button._counter = 0
            except Exception:
                pass

        upload_status.value = (
            "<span style='color:#555;'>"
            "No CSV file uploaded."
            "</span>"
        )

        input_status.value = ""

        with output:
            output.clear_output()
            print("CSV input cleared.")

    clear_button.on_click(
        clear_all
    )

    # ------------------------------------------------------------
    # Inspect data and display column selectors
    # ------------------------------------------------------------

    def start_processing(_):
        """
        Inspect the CSV and display column-selection controls.
        """

        with output:
            output.clear_output()

            text = input_box.value.strip()

            if not text:
                print(
                    "Upload a CSV file, paste CSV data, "
                    "or click Load Example."
                )
                return

            try:
                headers = inspect_csv_headers(
                    text
                )

                if len(headers) < 3:
                    raise ValueError(
                        "The CSV must contain at least three columns."
                    )

                (
                    latitude_default,
                    longitude_default,
                    time_default
                ) = determine_column_defaults(headers)

                latitude_dropdown = Dropdown(
                    options=headers,
                    value=latitude_default,
                    description="Latitude:",
                    layout=Layout(width="300px")
                )

                longitude_dropdown = Dropdown(
                    options=headers,
                    value=longitude_default,
                    description="Longitude:",
                    layout=Layout(width="300px")
                )

                time_dropdown = Dropdown(
                    options=headers,
                    value=time_default,
                    description="Timestamp:",
                    layout=Layout(width="300px")
                )

                calculate_button = Button(
                    description="Calculate Results",
                    button_style="success",
                    icon="check",
                    layout=Layout(width="190px")
                )

                method_label = get_method_label(
                    method_selector.value
                )

                print(f"Selected method: {method_label}")
                print(
                    "Confirm the latitude, longitude, and "
                    "timestamp columns below."
                )

                def really_process(_):
                    """
                    Run the complete calculation after the user
                    confirms the column selections.
                    """

                    with output:
                        output.clear_output()

                        calculate_button.disabled = True
                        calculate_button.description = "Processing..."

                        try:
                            process_csv(
                                text,
                                latitude_dropdown.value,
                                longitude_dropdown.value,
                                time_dropdown.value,
                                method_selector.value
                            )

                        except Exception as error:
                            print(f"Error: {error}")

                        finally:
                            calculate_button.disabled = False
                            calculate_button.description = (
                                "Calculate Results"
                            )

                calculate_button.on_click(
                    really_process
                )

                display(
                    VBox(
                        [
                            HTML(
                                "<h4 style='margin-bottom:4px;'>"
                                "Confirm CSV columns"
                                "</h4>"
                            ),
                            HBox(
                                [
                                    latitude_dropdown,
                                    longitude_dropdown,
                                    time_dropdown
                                ],
                                layout=Layout(
                                    flex_flow="row wrap",
                                    gap="8px"
                                )
                            ),
                            calculate_button
                        ],
                        layout=Layout(
                            border="1px solid #cccccc",
                            padding="12px",
                            width="100%"
                        )
                    )
                )

            except Exception as error:
                print(
                    f"Could not inspect the CSV: {error}"
                )

    process_button.on_click(
        start_processing
    )

    # ------------------------------------------------------------
    # Main display
    # ------------------------------------------------------------

    title = HTML(
        """
        <div style="margin-bottom:12px;">
            <h2 style="margin-bottom:5px;">
                GPS Distance and Speed Calculator
            </h2>
            <p style="margin-top:0;">
                Upload a CSV file or paste CSV content below.
                Uploaded file content is copied into the text area,
                allowing review and editing before processing.
            </p>
        </div>
        """
    )

    method_heading = HTML(
        """
        <h3 style="margin-bottom:4px;">
            1. Select distance method
        </h3>
        """
    )

    input_heading = HTML(
        """
        <h3 style="margin-bottom:4px;">
            2. Provide CSV data
        </h3>
        """
    )

    process_heading = HTML(
        """
        <h3 style="margin-bottom:4px;">
            3. Process data
        </h3>
        """
    )

    input_controls = HBox(
        [
            upload_button,
            example_button,
            clear_button
        ],
        layout=Layout(
            flex_flow="row wrap",
            gap="8px",
            width="100%"
        )
    )

    display(
        VBox(
            [
                title,
                method_heading,
                method_selector,
                input_heading,
                input_controls,
                upload_status,
                input_status,
                input_box,
                process_heading,
                process_button,
                help_section,
                output
            ],
            layout=Layout(
                width="100%",
                max_width="1200px"
            )
        )
    )


# ================================================================
# START APPLICATION
# ================================================================

run_with_ui()